Criar uma tabela de Clientes para equipe de marketing:<br>
Importante somente pessoas que tenham e-mail e tefefone cadastrados


In [0]:
# Definir pastas do projetos em variaveis para facilitar
bronze_path   = '/Volumes/bikestore/default/bikestore/bronze/'
silver_path   = '/Volumes/bikestore/default/bikestore/silver/'
gold_path     = '/Volumes/bikestore/default/bikestore/gold/'
resource_path = '/Volumes/bikestore/default/bikestore/resource/origem/'

In [0]:
#criando um dicionario com o caminho de cada pasta do arquivo parquet/delta
bronze_map = {
    #'tmp_brands': f'{bronze_path}brand/',
    'tmp_customers': f'{bronze_path}customers/',
    #'tmp_orders': f'{bronze_path}orders/',
    #'tmp_order_items': f'{bronze_path}orders_item/',
    #'tmp_products': f'{bronze_path}products/',
    #'tmp_stores': f'{bronze_path}stores/',
    #'tmp_staff': f'{bronze_path}staffs/',
    #'tmp_categories': f'{bronze_path}categories/',
    #'tmp_stocks': f'{bronze_path}stocks/'
    }

#fazendo um looping para criar uma tabela temporaria pra cada tabela
for key, value in bronze_map.items():
    (spark.read.format('delta')
        .load(value)
        .createOrReplaceTempView(key)
)

In [0]:
%python
df_customer_silver = spark.sql("""

SELECT 
  CT.customer_id
  ,CT.first_name
  ,CT.last_name
  ,CT.phone
  ,CT.email
  ,CT.street
  ,CT.city
  ,CT.state
  ,CT.zip_code
FROM  tmp_customers CT
WHERE  1=1
AND  CT.phone IS NOT NULL -- vazio 
AND  CT.phone NOT IN ('NULL','NULL ')-- texto null
and  CT.email IS NOT NULL -- vazio 
AND  CT.email NOT IN ('NULL','NULL ')
         
                              
                              """)

# salvar em Delta na silver 
df_customer_silver.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchema','true')\
    .save(f'{silver_path}customer')


In [0]:
df = df_customer_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bikestore.logistics.silver_customers")